In [9]:
import os
import glob
import numpy as np
import math
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import xml.etree.ElementTree as ET
from tqdm.notebook import tqdm
import pickle

# --- CONFIGURATION ---
CONFIG = {
    'xml_root': r'E:\DATA\Annotations',
    'video_root': r'E:\DATA\Videos',
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'target_label': 'bangla-tesla',
    'obs_len': 15,
    'pred_len': 45,
    'batch_size': 32,
    'epochs': 30,
    'lr': 0.0001, 
    'd_model': 64,
    'nhead': 4,
    'layers': 3
}
print(f"✅ Transformer Config Loaded. Device: {CONFIG['device']}")

✅ Transformer Config Loaded. Device: cpu


In [10]:
class BanglaTeslaDataset(Dataset):
    def __init__(self, xml_root, obs_len=15, pred_len=45, target_label='bangla-tesla'):
        self.seq_len = obs_len + pred_len
        self.samples = []
        xml_files = glob.glob(os.path.join(xml_root, '**', '*.xml'), recursive=True)
        
        for xml in tqdm(xml_files, desc="Parsing XMLs"):
            try:
                tree = ET.parse(xml)
                meta = tree.getroot().find('meta').find('original_size')
                img_w, img_h = float(meta.find('width').text), float(meta.find('height').text)
                
                for track in tree.getroot().findall('track'):
                    if track.attrib['label'] != target_label: continue
                    boxes = sorted(track.findall('box'), key=lambda b: int(b.attrib['frame']))
                    data = []
                    for box in boxes:
                        if box.get('outside') == '1': continue
                        xtl, ytl = float(box.attrib['xtl']), float(box.attrib['ytl'])
                        xbr, ybr = float(box.attrib['xbr']), float(box.attrib['ybr'])
                        data.append([(xtl+xbr)/(2*img_w), (ytl+ybr)/(2*img_h), (xbr-xtl)/img_w, (ybr-ytl)/img_h])
                    
                    data = np.array(data)
                    if len(data) < self.seq_len: continue
                    
                    for i in range(0, len(data) - self.seq_len + 1, 5):
                        self.samples.append({
                            'obs': data[i:i+obs_len],
                            'target': data[i+obs_len:i+obs_len+pred_len]
                        })
            except: pass

    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        return (torch.tensor(self.samples[idx]['obs'], dtype=torch.float32),
                torch.tensor(self.samples[idx]['target'], dtype=torch.float32))

dataset = BanglaTeslaDataset(CONFIG['xml_root'])
train_size = int(0.8 * len(dataset))
train_set, val_set = torch.utils.data.random_split(dataset, [train_size, len(dataset)-train_size])
train_loader = DataLoader(train_set, batch_size=CONFIG['batch_size'], shuffle=True)
val_loader = DataLoader(val_set, batch_size=CONFIG['batch_size'], shuffle=False)
print(f"✅ Data Ready: {len(train_set)} Train, {len(val_set)} Val")

Parsing XMLs:   0%|          | 0/6 [00:00<?, ?it/s]

✅ Data Ready: 3410 Train, 853 Val


In [11]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x): return x + self.pe[:, :x.size(1), :]

class TrajTransformer(nn.Module):
    def __init__(self, d_model=64, nhead=4, layers=3):
        super().__init__()
        self.input_fc = nn.Linear(4, d_model)
        self.pos_enc = PositionalEncoding(d_model)
        self.transformer = nn.Transformer(d_model, nhead, layers, layers, batch_first=True)
        self.out_fc = nn.Linear(d_model, 4)

    def generate_mask(self, sz):
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask

    def forward(self, src, tgt):
        src = self.pos_enc(self.input_fc(src))
        tgt = self.pos_enc(self.input_fc(tgt))
        tgt_mask = self.generate_mask(tgt.size(1)).to(src.device)
        out = self.transformer(src, tgt, tgt_mask=tgt_mask)
        return self.out_fc(out)

model = TrajTransformer(CONFIG['d_model'], CONFIG['nhead'], CONFIG['layers']).to(CONFIG['device'])
optimizer = optim.Adam(model.parameters(), lr=CONFIG['lr'])
criterion = nn.MSELoss()
print("✅ Transformer Initialized")

✅ Transformer Initialized


In [12]:
print("🚀 Training Transformer...")
history = {'train_loss': [], 'val_loss': []}

for epoch in range(CONFIG['epochs']):
    # --- TRAIN ---
    model.train()
    train_loss_ep = 0
    for obs, target in tqdm(train_loader, leave=False, desc=f"Epoch {epoch+1} Train"):
        obs, target = obs.to(CONFIG['device']), target.to(CONFIG['device'])
        optimizer.zero_grad()
        
        # Teacher Forcing
        start_token = obs[:, -1, :].unsqueeze(1)
        dec_input = torch.cat([start_token, target[:, :-1, :]], dim=1)
        
        preds = model(obs, dec_input)
        loss = criterion(preds, target)
        loss.backward()
        optimizer.step()
        train_loss_ep += loss.item()
    
    # --- VALIDATION ---
    model.eval()
    val_loss_ep = 0
    with torch.no_grad():
        for obs, target in val_loader:
            obs, target = obs.to(CONFIG['device']), target.to(CONFIG['device'])
            
            # Same Teacher Forcing strategy for comparable Loss
            start_token = obs[:, -1, :].unsqueeze(1)
            dec_input = torch.cat([start_token, target[:, :-1, :]], dim=1)
            
            preds = model(obs, dec_input)
            loss = criterion(preds, target)
            val_loss_ep += loss.item()

    avg_train = train_loss_ep / len(train_loader)
    avg_val = val_loss_ep / len(val_loader)
    history['train_loss'].append(avg_train)
    history['val_loss'].append(avg_val)
    
    print(f"Epoch {epoch+1}/{CONFIG['epochs']} | Train Loss: {avg_train:.6f} | Val Loss: {avg_val:.6f}")

torch.save(model.state_dict(), "bangla_tesla_transformer.pth")

🚀 Training Transformer...


Epoch 1 Train:   0%|          | 0/107 [00:00<?, ?it/s]

Epoch 1/30 | Train Loss: 0.034850 | Val Loss: 0.004997


Epoch 2 Train:   0%|          | 0/107 [00:00<?, ?it/s]

Epoch 2/30 | Train Loss: 0.008879 | Val Loss: 0.002473


...

Epoch 28/30 | Train Loss: 0.000666 | Val Loss: 0.000235


Epoch 29 Train:   0%|          | 0/107 [00:00<?, ?it/s]

Epoch 29/30 | Train Loss: 0.000640 | Val Loss: 0.000178


Epoch 30 Train:   0%|          | 0/107 [00:00<?, ?it/s]

Epoch 30/30 | Train Loss: 0.000606 | Val Loss: 0.000210


In [13]:
import numpy as np

def calculate_metrics_transformer(model, loader):
    model.eval()
    mse_list = []
    cmse_list = []
    cfmse_list = []
    
    # Resolution for Un-normalization
    W_px, H_px = 2592, 1944
    
    print("📊 Generating Autoregressive Predictions...")
    with torch.no_grad():
        for obs, target in tqdm(loader, leave=False):
            obs = obs.to(CONFIG['device'])
            target = target.cpu().numpy()
            
            # --- Autoregressive Loop ---
            # 1. Start with the last observation frame
            curr_step = obs[:, -1, :].unsqueeze(1) # [Batch, 1, 4]
            dec_input = curr_step
            
            generated_seq = []
            
            for _ in range(CONFIG['pred_len']):
                # Predict next step based on history + generated so far
                out = model(obs, dec_input)
                
                # The model outputs the whole sequence, we take the last generated step
                next_step = out[:, -1, :].unsqueeze(1)
                
                generated_seq.append(next_step)
                
                # Append to input for next iteration
                dec_input = torch.cat([dec_input, next_step], dim=1)
            
            # Concatenate to get shape [Batch, 45, 4]
            preds = torch.cat(generated_seq, dim=1).cpu().numpy()
            
            # --- Metric Calculation ---
            
            # 1. Un-normalize Coordinates
            # Center X, Width -> Scale by Width
            # Center Y, Height -> Scale by Height
            p_cx = preds[:,:,0] * W_px; t_cx = target[:,:,0] * W_px
            p_cy = preds[:,:,1] * H_px; t_cy = target[:,:,1] * H_px
            p_w  = preds[:,:,2] * W_px; t_w  = target[:,:,2] * W_px
            p_h  = preds[:,:,3] * H_px; t_h  = target[:,:,3] * H_px
            
            # 2. MSE (Average over all 45 frames)
            # dist = (x-x)^2 + (y-y)^2
            frame_sq_err = (p_cx - t_cx)**2 + (p_cy - t_cy)**2
            mse_list.extend(np.mean(frame_sq_err, axis=1))
            
            # 3. C-MSE (Center Error at Last Frame)
            # Take index -1
            final_sq_err = (p_cx[:,-1] - t_cx[:,-1])**2 + (p_cy[:,-1] - t_cy[:,-1])**2
            cmse_list.extend(final_sq_err)
            
            # 4. CF-MSE (Center + Foot Error at Last Frame)
            # Foot Y = Center Y + (Height / 2)
            p_foot_y = p_cy[:,-1] + (p_h[:,-1] / 2)
            t_foot_y = t_cy[:,-1] + (t_h[:,-1] / 2)
            
            # Foot Error
            foot_sq_err = (p_cx[:,-1] - t_cx[:,-1])**2 + (p_foot_y - t_foot_y)**2
            
            # CF = Center Error + Foot Error
            cfmse_list.extend(final_sq_err + foot_sq_err)

    return np.mean(mse_list), np.mean(cmse_list), np.mean(cfmse_list)

mse, cmse, cfmse = calculate_metrics_transformer(model, val_loader)

print("-" * 30)
print(f"🏆 Transformer Results (Pixels):")
print(f"   MSE (Avg Trajectory): {mse:.2f}")
print(f"   C-MSE (Final Center): {cmse:.2f}")
print(f"   CF-MSE (Center+Foot): {cfmse:.2f}")
print("-" * 30)

📊 Generating Autoregressive Predictions...


  0%|          | 0/27 [00:00<?, ?it/s]

------------------------------
🏆 Transformer Results (Pixels):
   MSE (Avg Trajectory): 43679.06
   C-MSE (Final Center): 109290.25
   CF-MSE (Center+Foot): 221782.73
------------------------------
